In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv3D, BatchNormalization, MaxPooling3D, Dropout, GlobalAveragePooling3D, Dense, Concatenate, Add, Activation
from tensorflow.keras.optimizers import Adam

import nibabel as nib
import scipy.ndimage as ndimage
import tensorflow.keras.backend as K

import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras.metrics import Precision, Recall, AUC, Accuracy
import gc

from tensorflow.keras import mixed_precision

In [ ]:
import random
random.seed(1234)
np.random.seed(1234)
tf.random.set_seed(1234)

In [ ]:
# Set global data format to 'channels_first'
K.set_image_data_format('channels_first')

# Data generator class for loading data in batches
class DataGenerator(keras.utils.Sequence):
    def __init__(self, df, batch_size=8, dim=(64, 64, 64), n_channels=1, n_classes=2, shuffle=True):
        self.dim = dim
        self.batch_size = batch_size
        self.df = df  # DataFrame containing patient IDs, labels, and image paths
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        # Calculate the number of batches per epoch
        return int(np.floor(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        # Generate a batch of data
        indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        batch_df = self.df.iloc[indexes]
        X, y = self.__data_generation(batch_df)
        return X, y

    def on_epoch_end(self):
        # Update indices after each epoch
        self.indexes = np.arange(len(self.df))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __changeshape(self, imgs, new_shape):
        # Rescale image to the target shape
        if len(imgs.shape) == 3:
            resize_factor = [a / b for a, b in zip(new_shape, imgs.shape)]
            imgnew = zoom(imgs, resize_factor, mode='nearest')
            return imgnew
        else:
            raise ValueError('Wrong shape, expected 3D array')  

    def normalize(self, volume):    
        window_level = -600
        window_width = 1600
        min = window_level - window_width // 2  # -600 - 1500//2 = -1350
        max = window_level + window_width // 2  # -600 + 1500//2 = 150
        volume[volume < min] = min
        volume[volume > max] = max
        volume = (volume - min) / (max - min)
        volume = volume.astype("float32")
        return volume
    
    def __data_generation(self, batch_df):
        # Generate a batch of data
        X_pre = np.empty((self.batch_size, self.n_channels, *self.dim), dtype=np.float32)
        X_post = np.empty((self.batch_size, self.n_channels, *self.dim), dtype=np.float32)
        y = np.empty((self.batch_size), dtype=int)
        
        for i, row in enumerate(batch_df.itertuples()):
            # Read Pre image data
            pre_path = row.Pre_path
            pre_img = nib.load(pre_path).get_fdata()
            pre_reshaped = self.__changeshape(pre_img, self.dim)
            pre_normalized = self.normalize(pre_reshaped)
            X_pre[i, 0] = pre_normalized
            
            # Read Post image data
            pseudo_path = row.Post_pseudo_path
            pseudo_img = nib.load(pseudo_path).get_fdata()
            pseudo_reshaped = self.__changeshape(pseudo_img, self.dim)
            pseudo_normalized = self.normalize(pseudo_reshaped)
            X_post[i, 0] = pseudo_normalized
            
            y[i] = getattr(row, 'Binary_Label')

        return [X_pre, X_post], tf.keras.utils.to_categorical(y, num_classes=self.n_classes)

In [ ]:
def get_predictions_and_labels(generator, model):
    predictions = []
    labels = []
    for batch in generator:
        X, y = batch
        # Get the model's predicted probability for the positive class (class 1)
        preds = model.predict(X)[:, 1]  
        predictions.extend(preds)
        # Convert one-hot encoded labels to regular class labels
        labels.extend(np.argmax(y, axis=1))  # y is one-hot encoded
    return np.array(predictions), np.array(labels)


# Load training and validation data splits for each fold
def load_train_val_split(fold_dir):
    train_split = pd.read_csv(fold_dir + 'train_split.csv')
    val_split = pd.read_csv(fold_dir + 'val_split.csv')
    return train_split, val_split

# Load the test split
def load_test_split(test_file):
    test_split = pd.read_csv(test_file)
    return test_split

# Function to save metrics
def save_metrics(fold_num, train_metrics, val_metrics, test_metrics, history_metrics):
    # Save performance metrics (train, validation, test) for each fold
    fold_metrics = {
        'Fold': fold_num,
        'Train_ROC': round(train_metrics['ROC'], 3),
        'Train_Accuracy': round(train_metrics['Accuracy'], 3),
        'Train_Precision': round(train_metrics['Precision'], 3),
        'Train_Recall': round(train_metrics['Recall'], 3),
        'Train_F1 Score': round(train_metrics['F1 Score'], 3),
        'Val_ROC': round(val_metrics['ROC'], 3),
        'Val_Accuracy': round(val_metrics['Accuracy'], 3),
        'Val_Precision': round(val_metrics['Precision'], 3),
        'Val_Recall': round(val_metrics['Recall'], 3),
        'Val_F1 Score': round(val_metrics['F1 Score'], 3),
        'Test_ROC': round(test_metrics['ROC'], 3),
        'Test_Accuracy': round(test_metrics['Accuracy'], 3),
        'Test_Precision': round(test_metrics['Precision'], 3),
        'Test_Recall': round(test_metrics['Recall'], 3),
        'Test_F1 Score': round(test_metrics['F1 Score'], 3)
    }
    history_metrics.append(fold_metrics)

# Function to calculate metrics
def calculate_metrics(predictions, labels):
    metrics = {}
    metrics['ROC'] = roc_auc_score(labels, predictions)
    metrics['Accuracy'] = accuracy_score(labels, predictions.round())
    metrics['Precision'] = precision_score(labels, predictions.round())
    metrics['Recall'] = recall_score(labels, predictions.round())
    metrics['F1 Score'] = f1_score(labels, predictions.round())
    return metrics

# Save predictions to a CSV file
def save_predictions(fold_dir, filename, predictions, labels):
    data = {'Predictions': predictions, 'Labels': labels}
    df_save = pd.DataFrame(data)
    df_save.to_csv(os.path.join(fold_dir, filename), index=False)


In [ ]:
# CNN Model Builder
def build_CNN(input_size, weight_decay, dropout_rate, class_num):
    input_layer = Input([1, input_size[0], input_size[1], input_size[2]])
    x = Conv3D(32, (3, 3, 3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(weight_decay))(input_layer)
    x = BatchNormalization()(x)
    x = MaxPooling3D(pool_size=(2, 2, 2))(x)
    
    x = Conv3D(64, (3, 3, 3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = BatchNormalization()(x)
    x = MaxPooling3D(pool_size=(2, 2, 2))(x)
    
    x = Conv3D(128, (3, 3, 3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = BatchNormalization()(x)
    x = MaxPooling3D(pool_size=(2, 2, 2))(x)
    
    x = Dropout(dropout_rate)(x)
    x = GlobalAveragePooling3D()(x)
    predictions = Dense(class_num, activation='softmax')(x)
    
    model_CNN = Model(inputs=input_layer, outputs=predictions)
    return model_CNN

In [ ]:
# Base CNN for Siamese Network
def build_base_CNN(input_size, weight_decay, dropout_rate):
    input_layer = Input([1, input_size[0], input_size[1], input_size[2]])
    
    x = Conv3D(32, (3, 3, 3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(weight_decay))(input_layer)
    x = BatchNormalization()(x)
    x = MaxPooling3D(pool_size=(2, 2, 2), padding='same')(x)
    
    x = Conv3D(64, (3, 3, 3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = BatchNormalization()(x)
    x = MaxPooling3D(pool_size=(2, 2, 2), padding='same')(x)
    
    x = Conv3D(128, (3, 3, 3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = BatchNormalization()(x)
    x = MaxPooling3D(pool_size=(2, 2, 2), padding='same')(x)
    
    model = Model(inputs=input_layer, outputs=x)
    return model

# Siamese CNN Model Builder ：STRANet
def build_siamese_CNN(input_size, weight_decay, dropout_rate, class_num):
    input_shape = (1, input_size[0], input_size[1], input_size[2])
    
    # Build base shared CNN model
    base_model = build_base_CNN(input_size, weight_decay, dropout_rate)
    
    input_a = Input(shape=input_shape, name='input_a')
    input_b = Input(shape=input_shape, name='input_b')
    
    processed_a = base_model(input_a)
    processed_b = base_model(input_b)
    
    concatenated = Concatenate(axis=1, name='concat')([processed_a, processed_b])
    
    x = GlobalAveragePooling3D(name='global_avg_pool')(concatenated)
    x = Dense(128, activation='relu', name='dense_128')(x)
    x = Dropout(dropout_rate, name='dropout')(x)
    predictions = Dense(class_num, activation='softmax', name='output_layer')(x)
    
    siamese_model = Model(inputs=[input_a, input_b], outputs=predictions)
    return siamese_model


In [ ]:
# DenseNet Model Builder
def dense_block(x, num_layers, growth_rate, weight_decay):
    for i in range(num_layers):
        conv = Conv3D(growth_rate, (3, 3, 3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(weight_decay))(x)
        conv = BatchNormalization()(conv)
        x = Concatenate(axis=1)([x, conv])  # Concatenate along channels axis
    return x

def transition_layer(x, reduction_factor, weight_decay):
    num_filters = int(x.shape[1] * reduction_factor)
    x = Conv3D(num_filters, (1, 1, 1), padding='same', activation='relu', kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = BatchNormalization()(x)
    x = MaxPooling3D(pool_size=(2, 2, 2))(x)
    return x

def build_model_densenet(input_size, weight_decay, dropout_rate, growth_rate=32, num_layers_per_block=2, reduction_factor=0.5):
    input_layer = Input([1, input_size[0], input_size[1], input_size[2]])

    x = Conv3D(64, (7, 7, 7), padding='same', activation='relu', kernel_regularizer=regularizers.l2(weight_decay))(input_layer)
    x = BatchNormalization()(x)
    x = MaxPooling3D(pool_size=(2, 2, 2))(x)

    for _ in range(2):  # Dense blocks
        x = dense_block(x, num_layers=num_layers_per_block, growth_rate=growth_rate, weight_decay=weight_decay)
        x = transition_layer(x, reduction_factor, weight_decay)
    
    x = GlobalAveragePooling3D()(x)

    model_densenet = Model(inputs=input_layer, outputs=x)
    return model_densenet

# Siamese DenseNet Model Builder
def build_siamese_densenet(input_size, weight_decay, dropout_rate, class_num):
    input_shape = (1, input_size[0], input_size[1], input_size[2])

    base_model = build_model_densenet(input_size, weight_decay, dropout_rate)

    input_a = Input(shape=input_shape, name='input_a')
    input_b = Input(shape=input_shape, name='input_b')

    processed_a = base_model(input_a)
    processed_b = base_model(input_b)

    concatenated = Concatenate(axis=1, name='concat')([processed_a, processed_b])

    x = Dense(128, activation='relu', name='dense_128')(concatenated)
    x = Dropout(dropout_rate, name='dropout')(x)
    predictions = Dense(class_num, activation='softmax', name='output_layer')(x)

    siamese_densenet = Model(inputs=[input_a, input_b], outputs=predictions)
    return siamese_densenet


In [ ]:
# 3D ResNet Block
def resnet_block_3d(x, filters, kernel_size=3, stride=1, weight_decay=1e-4):
    shortcut = x
    x = Conv3D(filters, kernel_size, strides=stride, padding='same', kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv3D(filters, kernel_size, strides=1, padding='same', kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = BatchNormalization()(x)
    if shortcut.shape[-1] != x.shape[-1]:
        shortcut = Conv3D(filters, kernel_size=1, strides=stride, padding='same', kernel_regularizer=regularizers.l2(weight_decay))(shortcut)
        shortcut = BatchNormalization()(shortcut)
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

# Build 3D ResNet Model
def build_3D_ResNet(input_size, weight_decay):
    input_layer = Input([1, input_size[0], input_size[1], input_size[2]])

    x = Conv3D(64, (7, 7, 7), strides=2, padding='same', kernel_regularizer=regularizers.l2(weight_decay))(input_layer)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling3D(pool_size=(3, 3, 3), strides=2, padding='same')(x)

    x = resnet_block_3d(x, 64)
    x = resnet_block_3d(x, 64)
    x = resnet_block_3d(x, 128, stride=2)
    x = resnet_block_3d(x, 128)
    x = resnet_block_3d(x, 256, stride=2)
    x = resnet_block_3d(x, 256)
    x = resnet_block_3d(x, 512, stride=2)
    x = resnet_block_3d(x, 512)

    x = GlobalAveragePooling3D()(x)

    model_3D_ResNet18 = Model(inputs=input_layer, outputs=x)
    return model_3D_ResNet18

# Siamese Network using 3D ResNet
def build_siamese_resnet(input_size, weight_decay, dropout_rate, class_num):
    input_a = Input([1, input_size[0], input_size[1], input_size[2]])
    input_b = Input([1, input_size[0], input_size[1], input_size[2]])

    base_network = build_3D_ResNet(input_size, weight_decay)

    output_a = base_network(input_a)
    output_b = base_network(input_b)

    concatenated = Concatenate(axis=1)([output_a, output_b])

    x = Dense(128, activation='relu')(concatenated)
    x = Dropout(dropout_rate)(x)
    predictions = Dense(class_num, activation='softmax')(x)

    siamese_resnet = Model(inputs=[input_a, input_b], outputs=predictions)
    return siamese_resnet
